# CASMI 2026 · RL77 Notebook 3 — Four-Way Ensemble (GLMR + Siamese ECFP4 + Mass2SMILES)

*Assembled from **CASMI 2026 Colab Rubric.pdf** (pages 27–38). Python cells are the reviewed `experiments/runs/RL77/nb3/cell_NN.py` files; text cells are taken from the PDF.*

> **User prompt:** Write the PyTorch module for the 2000-bin Siamese ECFP4 fingerprint retriever and Mass2SMILES 60-functional-group head, and show how to blend their similarity scores with GLMR Stage 2 re-ranking.

To push your **Enveda CASMI 2026** pipeline past the **0.490+ MRR@25** top-10% threshold, blending **GLMR** with the **2000-bin Siamese ECFP4 metric learner** (Carrera et al., BIOSTEC 2026) and the **Mass2SMILES 60-functional-group TCN head** (Elser et al.) introduces two orthogonal structural signals:

1. **Topological Fingerprint Recall (Siamese 2000-Bin ECFP4):** While autoregressive SMILES decoders can suffer from low topological Tanimoto similarity on sparse or noisy spectra, a 3-layer Siamese MLP trained on a dense 2000-bin spectral vector and 2048-bit Morgan (ECFP4) fingerprints achieves a retrieval AUC of 0.9947 and nearly doubles Top-1 Tanimoto fidelity (0.351 vs. 0.189).
2. **Substructural Isomer Discrimination (Mass2SMILES 60-FG + Heteroatoms):** Elser et al. demonstrated that functional group count accuracy does not strictly correlate with global Tanimoto similarity—meaning candidates with tied ChemFormer or ECFP4 cosine scores can be separated by penalizing candidates whose RDKit functional group, heteroatom, or glycoside/sugar counts conflict with the MS/MS fragmentation pattern.

## Prerequisites

| Input | Kaggle source | Provides |
| --- | --- | --- |
| Competition data | `enveda-CASMI26-molecule-id-mass-spectra` | `train.parquet`, `test.parquet`, `sample_submission.csv` |
| RL77 prerequisites | `scottweeden/casmi26-rl77-prereqs` | `rl77_env.py`, `rl77_weights.py`, `rl77_weights.json` (Evidently snapshot `rl77-v014`), upstream modules |
| Notebook 1 outputs | `scottweeden/casmi26-rl77-nb1-glmr-outputs` | Two-stage GLMR checkpoint, retrieval library, validation split, submission |
| Notebook 2 outputs | `scottweeden/casmi26-rl77-nb2-store-outputs` | Mass-sorted candidate catalog, FP16 ChemFormer memmap, FAISS index, formula index, offline wheels |

- **Order:** Run notebooks 1, 2 first and publish their outputs (`tools/kaggle_pipeline.py publish-output`).
- **Accelerator:** CPU (no GPU quota used); internet on for the test run so `ensure_packages()` can reach PyPI. Offline runs install from the wheels in notebook 2's output dataset.
- **Packages:** `rdkit==2026.03.3` (the metric's canonicalizer) and `faiss-cpu` are installed if the image lacks them; `torch`, `pyarrow`, `pandas`, `scipy` come from the Kaggle image.
- **Weights:** all blend weights, reward coefficients, and tolerances come from Evidently dataset `casmi-rl77-weights` (version `rl77-v014`, id `01a1ee49-89c9-7ce4-b327-df8b7cf1d48f`) through `W("...")`.
- **Outputs:** everything written to `/kaggle/working` is published as `scottweeden/casmi26-rl77-nb3-ensemble-outputs` for the next notebook.

In [ ]:
# Prerequisites: locate the RL77 helper modules, install pinned packages, load the Evidently weights snapshot
import glob, os, sys
from pathlib import Path

_lib_hits = sorted(glob.glob("/kaggle/input/**/rl77_env.py", recursive=True))
if _lib_hits:
    RL77_LIB = Path(_lib_hits[0]).parent
elif os.environ.get("RL77_LIB_DIR"):
    RL77_LIB = Path(os.environ["RL77_LIB_DIR"])  # e.g. experiments/runs/RL77/build/datasets/casmi26-rl77-prereqs
else:
    raise FileNotFoundError("Attach the scottweeden/casmi26-rl77-prereqs dataset or set RL77_LIB_DIR.")
sys.path.insert(0, str(RL77_LIB))

import rl77_env as ENV
ENV.ensure_packages()                      # rdkit==2026.03.3 + faiss-cpu (offline wheels when attached)
from rl77_weights import load_rl77_weights
W = load_rl77_weights()                    # every blend weight / reward coefficient in this notebook reads W("...")
from rl77_nb1_glmr import *  # nb1: CASMI26 RL77 NB1 GLMR
from rl77_nb2_store import *  # nb2: CASMI26 RL77 NB2 Offline Store
ENV.log(f"Competition data: {ENV.competition_dir()} | outputs: {ENV.work_dir()}")


## Cell 1: 2000-Bin Spectral Vectorizer, 2048-Bit ECFP4 & 71-D Mass2SMILES Target Extractor

Following **Mass2SMILES** (Figure 1 of Elser et al.), the auxiliary target vector has **71 dimensions**: 60 deterministic RDKit functional group counts (`rdkit.Chem.Fragments`), 8 heteroatom counts (N, O, S, P, F, Cl, Br, I), and 3 sugar/glycoside and ring-system counts.

> **Review fix:** `AllChem.GetMorganFingerprintAsBitVect` is deprecated in current RDKit; the Morgan generator yields the same radius-2 / 2048-bit fingerprint. `aggregate_molecule_2000_bins` weighted spectra by `log10(max(base_peak_intensity, 10))`, and `max(nan, 10)` is `nan` for the 46% of train rows with a null base peak, which turned the whole composite vector into NaN.

In [ ]:
# [Colab Ensemble Cell 1] 2000-Bin Spectral Vectorizer, ECFP4 (2048-bit), & 71-D Functional Group Extractor
import inspect
from functools import lru_cache
from typing import Tuple, List, Dict, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Fragments, rdMolDescriptors, rdFingerprintGenerator

# Deterministically collect the first 60 RDKit functional group fragment functions (fr_*, sorted by name)
RDKIT_FG_FUNCTIONS = sorted(
    [
        (name, func)
        for name, func in inspect.getmembers(Fragments, inspect.isfunction)
        if name.startswith("fr_")
    ],
    key=lambda x: x[0],
)[:60]

HETEROATOM_SYMBOLS = ["N", "O", "S", "P", "F", "Cl", "Br", "I"]

# Approximate oxygen/nitrogen-rich pyranose/furanose glycoside SMARTS for fast sugar moiety counting
SUGAR_MOIETY_SMARTS = Chem.MolFromSmarts("[OX2r5,OX2r6]1[CX4r5,CX4r6]([OX2H1,OX2])[CX4r5,CX4r6]([OX2H1,OX2])[CX4r5,CX4r6]([OX2H1,OX2])[CX4r5,CX4r6]1")


def spectrum_to_2000_bins(
    mzs: np.ndarray,
    intensities: np.ndarray,
    min_mz: float = 50.0,
    max_mz: float = 1250.0,
    num_bins: int = 2000,
) -> np.ndarray:
    """
    Bins a cleaned MS/MS peak list into a 2000-dimensional normalized intensity vector
    (Carrera et al., BIOSTEC 2026 Section 3.3).
    """
    binned = np.zeros(num_bins, dtype=np.float32)
    if len(mzs) == 0:
        return binned

    mzs = np.asarray(mzs, dtype=np.float32)
    intensities = np.asarray(intensities, dtype=np.float32)
    max_i = np.max(intensities)
    if max_i > 0:
        intensities = intensities / max_i

    valid = (mzs >= min_mz) & (mzs < max_mz) & (intensities >= 0.005)
    mzs, intensities = mzs[valid], np.sqrt(intensities[valid])

    bin_indices = ((mzs - min_mz) / (max_mz - min_mz) * num_bins).astype(np.int32)
    bin_indices = np.clip(bin_indices, 0, num_bins - 1)

    # Max-pool intensities within each mass bin
    np.maximum.at(binned, bin_indices, intensities)
    norm = np.linalg.norm(binned)
    if norm > 0:
        binned = binned / norm
    return binned


def aggregate_molecule_2000_bins(
    spectra_records: List[ dict ],
    num_bins: int = 2000,
) -> np.ndarray:
    """
    Pools multiple collision-energy spectra for a single molecule_id into a composite
    2000-bin vector weighted by log10(base_peak_intensity).
    """
    if not spectra_records:
        return np.zeros(num_bins, dtype=np.float32)

    composite = np.zeros(num_bins, dtype=np.float32)
    total_weight = 0.0
    for row in spectra_records:
        vec = spectrum_to_2000_bins(
            row["ms2_mzs"], row["ms2_normalized_intensities"], num_bins=num_bins
        )
        bpi = row.get("base_peak_intensity", 1e4)
        bpi = float(bpi) if bpi is not None and np.isfinite(float(bpi)) else 1e4  # 46% of train values are null
        bpi = max(bpi, 10.0)
        w = float(np.log10(bpi))
        composite += w * vec
        total_weight += w

    if total_weight > 0:
        composite /= total_weight
    norm = np.linalg.norm(composite)
    return composite / norm if norm > 0 else composite


@lru_cache(maxsize=8)
def _morgan_generator(radius: int, n_bits: int):
    return rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)


def smiles_to_ecfp4_and_mass2smiles_71d(
    smiles: str,
    n_bits: int = 2048,
    radius: int = 2,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Extracts:
      1. 2048-bit Morgan (ECFP4, radius=2) binary fingerprint (Carrera et al.).
      2. 71-dimensional scaled structural vector (Elser et al., Mass2SMILES):
         - Indices 0..59:  60 RDKit functional group counts
         - Indices 60..67: 8 heteroatom counts (N, O, S, P, F, Cl, Br, I)
         - Indices 68..70: 3 sugar/ring counts (sugar rings, aromatic rings, aliphatic rings)
    """
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    if mol is None:
        return None, None

    # 1. 2048-bit ECFP4 fingerprint
    # (AllChem.GetMorganFingerprintAsBitVect is deprecated in current RDKit; the generator gives the same bits.)
    ecfp4_arr = _morgan_generator(radius, n_bits).GetFingerprintAsNumPy(mol).astype(np.float32)

    # 2. Mass2SMILES 71-d functional group + heteroatom + sugar descriptor vector
    fg_71 = np.zeros(71, dtype=np.float32)
    for idx, (_, fg_fn) in enumerate(RDKIT_FG_FUNCTIONS):
        try:
            fg_71[idx] = float(fg_fn(mol))
        except Exception:
            fg_71[idx] = 0.0

    # Heteroatom counts (indices 60..67)
    atom_counts = {sym: 0 for sym in HETEROATOM_SYMBOLS}
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        if sym in atom_counts:
            atom_counts[sym] += 1
    for j, sym in enumerate(HETEROATOM_SYMBOLS):
        fg_71[60 + j] = float(atom_counts[sym])

    # Sugar moiety count & ring counts (indices 68..70)
    sugar_hits = len(mol.GetSubstructMatches(SUGAR_MOIETY_SMARTS)) if SUGAR_MOIETY_SMARTS else 0
    fg_71[68] = float(sugar_hits)
    fg_71[69] = float(rdMolDescriptors.CalcNumAromaticRings(mol))
    fg_71[70] = float(rdMolDescriptors.CalcNumAliphaticRings(mol))

    # Log1p scaling stabilizes regression across high-count functional groups & heteroatoms
    fg_71_scaled = np.log1p(np.clip(fg_71, 0.0, None))
    return ecfp4_arr, fg_71_scaled

## Cell 2: 2000-Bin Siamese ECFP4 Metric Learner (`SiameseECFP4Retriever`)

Following Table 2 and Figure 3 of **Carrera et al. (BIOSTEC 2026)**, this module pairs twin 3-layer ReLU MLPs that project the 2000-bin spectrum ($\mathbf{z}_s \in \mathbb{R}^{256}$) and 2048-bit ECFP4 fingerprint ($\mathbf{z}_f \in \mathbb{R}^{256}$) into a shared 256-dimensional cosine embedding space, augmented with a direct 2048-bit continuous Morgan fingerprint decoder head for differentiable Tanimoto scoring.

> **Review fix:** loss margin, temperature, and fingerprint weight come from the Evidently weights (`nb3.siamese_loss.*`).

In [ ]:
# [Colab Ensemble Cell 2] 2000-Bin Siamese Metric Learning + Continuous ECFP4 Predictor
class SiameseECFP4Retriever(nn.Module):
    """
    Implements the Siamese Metric Learning + Fingerprint MLP architecture (Carrera et al., 2026):
      - Spectrum Branch: 3-layer MLP (2000 -> 1024 -> 512 -> 256, ReLU) producing L2-normalized z_s.
      - Fingerprint Branch: 3-layer MLP (2048 -> 1024 -> 512 -> 256, ReLU) producing L2-normalized z_f.
      - Direct ECFP4 Head: Predicts 2048-bit probabilities p_hat for continuous Tanimoto evaluation.
    """

    def __init__(
        self,
        spec_bins: int = 2000,
        fp_bits: int = 2048,
        d_embed: int = 256,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.spec_encoder = nn.Sequential(
            nn.Linear(spec_bins, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, d_embed),
        )

        self.fp_encoder = nn.Sequential(
            nn.Linear(fp_bits, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, d_embed),
        )

        # Auxiliary Fingerprint FFN head (Table 4 of Carrera et al.)
        self.ecfp4_head = nn.Sequential(
            nn.Linear(d_embed, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, fp_bits),
        )

    def encode_spectrum(self, binned_spec: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        raw_z_s = self.spec_encoder(binned_spec)
        z_s = F.normalize(raw_z_s, p=2, dim=-1)
        fp_logits = self.ecfp4_head(raw_z_s)
        return z_s, fp_logits

    def encode_fingerprint(self, ecfp4: torch.Tensor) -> torch.Tensor:
        raw_z_f = self.fp_encoder(ecfp4)
        return F.normalize(raw_z_f, p=2, dim=-1)

    def forward(
        self, binned_spec: torch.Tensor, ecfp4: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z_s, fp_logits = self.encode_spectrum(binned_spec)
        z_f = self.encode_fingerprint(ecfp4)
        return z_s, z_f, fp_logits


class SiameseContrastiveTanimotoLoss(nn.Module):
    """
    Combines Cosine Contrastive Metric Loss on (z_s, z_f) pairs with
    differentiable Tanimoto + Binary Cross-Entropy loss on the 2048-bit ECFP4 head.
    """

    def __init__(
        self,
        margin: float = W("nb3.siamese_loss.margin"),
        tau: float = W("nb3.siamese_loss.tau"),
        fp_weight: float = W("nb3.siamese_loss.fp_weight"),
    ):
        super().__init__()
        self.margin = margin
        self.tau = tau
        self.fp_weight = fp_weight

    @staticmethod
    def continuous_tanimoto_similarity(pred_probs: torch.Tensor, target_bits: torch.Tensor) -> torch.Tensor:
        # pred_probs: (B, 2048), target_bits: (B, 2048)
        intersection = torch.sum(pred_probs * target_bits, dim=-1)
        union = (
            torch.sum(pred_probs.pow(2), dim=-1)
            + torch.sum(target_bits.pow(2), dim=-1)
            - intersection
            + 1e-6
        )
        return intersection / union

    def forward(
        self,
        z_s: torch.Tensor,
        z_f: torch.Tensor,
        fp_logits: torch.Tensor,
        target_ecfp4: torch.Tensor,
    ) -> torch.Tensor:
        B = z_s.size(0)
        labels = torch.arange(B, device=z_s.device)

        # 1. Symmetric InfoNCE + Margin Cosine Contrastive Loss in 256-d Siamese space
        sim_matrix = torch.matmul(z_s, z_f.T)
        infonce_loss = 0.5 * (
            F.cross_entropy(sim_matrix / self.tau, labels)
            + F.cross_entropy(sim_matrix.T / self.tau, labels)
        )

        pos_sim = torch.diag(sim_matrix)
        neg_mask = ~torch.eye(B, dtype=torch.bool, device=z_s.device)
        hardest_neg = sim_matrix.masked_fill(~neg_mask, -1.0).max(dim=-1).values
        margin_loss = F.relu(hardest_neg - pos_sim + self.margin).mean()

        # 2. Direct 2048-bit ECFP4 BCE + (1 - Tanimoto) loss
        bce_loss = F.binary_cross_entropy_with_logits(fp_logits, target_ecfp4)
        pred_probs = torch.sigmoid(fp_logits)
        tanimoto_loss = 1.0 - self.continuous_tanimoto_similarity(pred_probs, target_ecfp4).mean()

        return infonce_loss + margin_loss + self.fp_weight * (bce_loss + tanimoto_loss)

## Cell 3: Mass2SMILES Dilated TCN & 60-Functional-Group Multi-Task Head

Following **Elser et al. (Mass2SMILES Figure 1 & Methods)**, this module processes the sinusoidal spectral peak sequence through a **Temporal Convolutional Network (TCN)** with `kernel_size=8` and 256 filters across exponentially dilated residual blocks, followed by 2 × 2 dense layers predicting the **71-dimensional functional group + heteroatom + sugar vector** and **512-dimensional continuous chemical descriptor**.

In [ ]:
# [Colab Ensemble Cell 3] Mass2SMILES Temporal Convolutional Network (TCN) + 71-D Functional Group Head
class DilatedTCNBlock(nn.Module):
    """
    Residual Temporal Convolutional Block with kernel_size=8 and 256 filters
    (Elser et al., Mass2SMILES Methods).
    """

    def __init__(
        self,
        channels: int = 256,
        kernel_size: int = 8,
        dilation: int = 1,
        dropout: float = 0.1,
    ):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(
            channels, channels, kernel_size=kernel_size, dilation=dilation, padding=padding
        )
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv1d(
            channels, channels, kernel_size=kernel_size, dilation=dilation, padding=padding
        )
        self.norm2 = nn.GroupNorm(8, channels)
        self.dropout = nn.Dropout(dropout)
        self.padding = padding

    def _chomp(self, x: torch.Tensor) -> torch.Tensor:
        return x[:, :, : -self.padding] if self.padding > 0 else x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, L)
        out = F.gelu(self.norm1(self._chomp(self.conv1(x))))
        out = self.dropout(out)
        out = F.gelu(self.norm2(self._chomp(self.conv2(out))))
        out = self.dropout(out)
        return F.gelu(x + out)


class Mass2SMILESFunctionalGroupHead(nn.Module):
    """
    Consumes multi-spectrum sinusoidal sequence states H^s_mol (B, P, 256) from the
    GLMR spectral encoder, passes them through a dilated TCN stack (kernel_size=8,
    256 filters), and outputs:
      1. fg_71_pred (B, 71): 60 RDKit functional groups + 8 heteroatoms + 3 sugar/ring counts
      2. cddd_512_pred (B, 512): Continuous structural descriptor embedding
    """

    def __init__(
        self,
        d_model: int = 256,
        kernel_size: int = 8,
        dilations: Tuple[int, ...] = (1, 2, 4, 8, 16),
        out_fg_dim: int = 71,
        out_desc_dim: int = 512,
    ):
        super().__init__()
        self.tcn_stack = nn.ModuleList(
            [
                DilatedTCNBlock(channels=d_model, kernel_size=kernel_size, dilation=d)
                for d in dilations
            ]
        )
        self.attn_pool = nn.Linear(d_model, 1)

        # 2x2 Dense layers producing outputs of shape 71 and 512 (Mass2SMILES Figure 1)
        self.fg_branch = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, out_fg_dim),
            nn.Softplus(),  # Non-negative log1p functional group & atom counts
        )
        self.desc_branch = nn.Sequential(
            nn.Linear(d_model, out_desc_dim),
            nn.LayerNorm(out_desc_dim),
            nn.GELU(),
            nn.Linear(out_desc_dim, out_desc_dim),
        )

    def forward(self, h_s_mol: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # h_s_mol: (B, P, 256) -> transpose to (B, 256, P) for 1D temporal convolutions
        x = h_s_mol.transpose(1, 2)
        for block in self.tcn_stack:
            x = block(x)
        x_seq = x.transpose(1, 2)  # (B, P, 256)

        weights = F.softmax(self.attn_pool(x_seq).squeeze(-1), dim=-1)  # (B, P)
        pooled = torch.sum(x_seq * weights.unsqueeze(-1), dim=1)         # (B, 256)

        fg_71_pred = self.fg_branch(pooled)
        cddd_512_pred = F.normalize(self.desc_branch(pooled), p=2, dim=-1)
        return fg_71_pred, cddd_512_pred


def mass2smiles_fg_loss(pred_fg: torch.Tensor, target_fg: torch.Tensor) -> torch.Tensor:
    """
    Joint MAE + MSE loss used to train Mass2SMILES functional group predictions
    (Elser et al., Figure 2C & 2D).
    """
    mae = F.l1_loss(pred_fg, target_fg)
    mse = F.mse_loss(pred_fg, target_fg)
    return mae + mse

## Cell 4: Four-Way Ensemble Score Blender & GLMR Stage 2 Re-Ranker

To prevent one model's raw score scale from dominating another, `CASMI26EnsembleReranker` computes four structural similarity scores across the $K = 40$ pre-retrieved candidates per `molecule_id`, applies per-molecule **z-score standardization** combined with **Reciprocal Rank Fusion (RRF)**, and optimizes the blending weights ($\mathbf{w}$) on your local `timsTOF` validation split before running strict `inchikey14` deduplication.

> **Review fix:** the five blend weights, `fg_gamma`, `rrf_k`, and the generative-weight redistribution (0.6 / 0.4) are read from Evidently dataset `casmi-rl77-weights` instead of literals. The out-of-fold optimizer ranks mass-consistent candidates first, matching inference.

> **Review fix (after the nb3 Kaggle run):** a single Nelder–Mead run from the prior stalled on the piecewise-constant MRR@25 surface. One cross-fit half stopped at 0.222 while the Siamese stream alone scored 0.369, so the cross-fitted blend (0.308) underperformed one of its own inputs. The optimizer now scores the prior, uniform and every single-stream vector, and searches from the two best with an explicit simplex step.

In [ ]:
# [Colab Ensemble Cell 4] Multi-Paradigm Blender: GLMR Stage 2 + Siamese ECFP4 + Mass2SMILES 60-FG
from scipy.optimize import minimize


class CASMI26EnsembleReranker:
    """
    Combines four orthogonal similarity signals for each candidate i in {1..K}:
      1. s_glmr_gen:  cos(E^m_+, E^m_i)           [GLMR Stage 2 Generative Unimodal Similarity]
      2. s_glmr_pre:  cos(E^s_mol, E^m_i)         [GLMR Stage 1 Contrastive Spectral Similarity]
      3. s_siamese:   0.5*cos(z_s, z_f,i) + 0.5*Tanimoto(p_hat, ecfp4_i) [2000-Bin Siamese ECFP4]
      4. s_fg_match:  exp(-gamma * ||fg_hat - fg_i||_1)                  [Mass2SMILES 71-D FG Kernel]
    """

    def __init__(
        self,
        weights: Optional[Tuple[float, float, float, float, float]] = None,
        fg_gamma: float = W("nb3.reranker.fg_gamma"),
        rrf_k: float = W("nb3.reranker.rrf_k"),
    ):
        if weights is None:  # Evidently snapshot (casmi-rl77-weights) instead of the PDF literals
            weights = tuple(W.vec("nb3.reranker", ["w_gen", "w_pre", "w_siamese", "w_fg", "w_rrf"]))
        # Weights: [w_glmr_gen, w_glmr_pre, w_siamese_ecfp4, w_mass2smiles_fg, w_rrf]
        self.weights = np.asarray(weights, dtype=np.float64)
        self.fg_gamma = fg_gamma
        self.rrf_k = rrf_k

    @staticmethod
    def _zscore_normalize(scores: np.ndarray) -> np.ndarray:
        if len(scores) <= 1:
            return scores
        std = float(np.std(scores))
        if std < 1e-7:
            return np.zeros_like(scores)
        return (scores - float(np.mean(scores))) / std

    def _reciprocal_rank_fusion(self, score_matrices: List[np.ndarray]) -> np.ndarray:
        """Computes consensus Reciprocal Rank Fusion (RRF) across the 4 scoring paradigms."""
        k_cands = len(score_matrices[0])
        rrf_scores = np.zeros(k_cands, dtype=np.float64)
        for s_vec in score_matrices:
            ranks = np.argsort(np.argsort(-s_vec)) + 1  # 1-indexed rank
            rrf_scores += 1.0 / (self.rrf_k + ranks)
        return self._zscore_normalize(rrf_scores)

    def score_candidates_for_molecule(
        self,
        s_glmr_gen: np.ndarray,      # (K,)
        s_glmr_pre: np.ndarray,      # (K,)
        s_siamese_ecfp4: np.ndarray, # (K,)
        s_fg_l1_dist: np.ndarray,    # (K,) mean L1 distance across 71 FG/heteroatom targets
        gen_smiles_valid: bool = True,
    ) -> np.ndarray:
        """Returns blended ranking score for all K candidates of a single molecule_id."""
        s_fg_sim = np.exp(-self.fg_gamma * s_fg_l1_dist)

        z_gen = self._zscore_normalize(s_glmr_gen) if gen_smiles_valid else np.zeros_like(s_glmr_pre)
        z_pre = self._zscore_normalize(s_glmr_pre)
        z_sia = self._zscore_normalize(s_siamese_ecfp4)
        z_fg = self._zscore_normalize(s_fg_sim)

        active_streams = [z_pre, z_sia, z_fg]
        if gen_smiles_valid:
            active_streams.append(z_gen)
        z_rrf = self._reciprocal_rank_fusion(active_streams)

        w0, w1, w2, w3, w4 = self.weights
        if not gen_smiles_valid:
            # Redistribute generative weight to Stage 1 and Siamese ECFP4 if beam SMILES failed RDKit check
            w1 += W("nb3.reranker.gen_fail_to_pre") * w0
            w2 += W("nb3.reranker.gen_fail_to_siamese") * w0
            w0 = 0.0

        return w0 * z_gen + w1 * z_pre + w2 * z_sia + w3 * z_fg + w4 * z_rrf

    def optimize_weights_oof(
        self,
        oof_records: List[Dict],
        ground_truth_ik14: Dict[str, str],
    ) -> np.ndarray:
        """
        Optimizes ensemble weights [w_gen, w_pre, w_sia, w_fg, w_rrf] on out-of-fold
        timsTOF validation molecules using Nelder-Mead simplex search to maximize MRR@25.
        """

        def neg_mrr25_objective(raw_w: np.ndarray) -> float:
            w = np.clip(raw_w, 0.0, 1.0)
            if np.sum(w) <= 1e-6:
                return 0.0
            self.weights = w / np.sum(w)

            rr_list = []
            for rec in oof_records:
                mol_id = rec["molecule_id"]
                true_ik14 = ground_truth_ik14.get(mol_id)
                if not true_ik14:
                    continue
                blended = self.score_candidates_for_molecule(
                    rec["s_glmr_gen"],
                    rec["s_glmr_pre"],
                    rec["s_siamese_ecfp4"],
                    rec["s_fg_l1_dist"],
                    gen_smiles_valid=rec["gen_valid"],
                )
                in_window = np.asarray(rec.get("in_window", np.zeros(len(blended), dtype=bool)), dtype=bool)
                order = np.lexsort((-blended, ~in_window))  # mass-consistent candidates first

                seen_ik14 = set()
                rank = 0
                matched_rr = 0.0
                for idx in order:
                    ik14 = rec["cand_ik14s"][idx]
                    if not ik14 or ik14 in seen_ik14:
                        continue
                    seen_ik14.add(ik14)
                    rank += 1
                    if ik14 == true_ik14:
                        matched_rr = 1.0 / rank
                        break
                    if rank >= 25:
                        break
                rr_list.append(matched_rr)

            return -float(np.mean(rr_list))

        # MRR@25 is piecewise constant in the weights, so one Nelder-Mead run from the prior can stall below
        # a trivial corner (Kaggle nb3 v1: one fold stopped at 0.222 while the Siamese stream alone scored ~0.37).
        # Score the prior, uniform and every single-stream vector, then search from the two best starts with an
        # explicit 0.2 simplex step (SciPy's default step for a zero entry is 0.00025, so one-hot starts never move).
        n = len(self.weights)
        starts = [np.asarray(self.weights, dtype=np.float64), np.full(n, 1.0 / n)] + [np.eye(n)[i] for i in range(n)]
        scored = sorted((neg_mrr25_objective(s), i) for i, s in enumerate(starts))  # ties keep the prior (index 0)
        best_fun, best_w = scored[0][0], starts[scored[0][1]]
        for _, i in scored[:2]:
            x0 = starts[i]
            simplex = np.vstack([x0] + [x0 + 0.2 * np.eye(n)[j] for j in range(n)])
            res = minimize(
                neg_mrr25_objective,
                x0,
                method="Nelder-Mead",
                options={"maxiter": 250, "xatol": 1e-3, "fatol": 1e-4, "initial_simplex": simplex},
            )
            if res.fun < best_fun:
                best_fun, best_w = res.fun, res.x
        best_w = np.clip(best_w, 0.0, 1.0)
        self.weights = best_w / np.sum(best_w)
        print(f"Optimized OOF MRR@25: {-best_fun:.4f} | Weights: {np.round(self.weights, 4)}")
        return self.weights

## Cell 5: Complete Integrated Inference Loop (GLMR + Siamese ECFP4 + Mass2SMILES)

Drop this function into your Kaggle inference notebook to execute all three models in a single pass per batch and produce your deduplicated `submission.csv`.

> **Review fix:** the loop read `batch["binned_2000"]`, which no dataset produced (KeyError on the first batch); `EnsembleMultiSpectrumDataset` adds it. Candidate fingerprints are cached, ranking puts mass-consistent candidates first, and per-candidate stream scores are recorded so Cell 6 can fit the blend out-of-fold. Retrieval uses the notebook-2 offline store (train + COCONUT).

In [ ]:
# [Colab Ensemble Cell 5] End-to-End Blended Inference & InChIKey14 Deduplication
class EnsembleMultiSpectrumDataset(MultiSpectrumMoleculeDataset):
    """Notebook-1 dataset plus the 2000-bin composite vector the Siamese branch reads as batch["binned_2000"].

    The PDF's inference loop used batch["binned_2000"], but no dataset produced it (KeyError on the first batch).
    """

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = super().__getitem__(idx)
        _, records = self.groups[idx]
        records = records[: self.max_spectra_per_mol]
        item["binned_2000"] = torch.from_numpy(aggregate_molecule_2000_bins(records))
        return item


def make_ensemble_loader(df: pd.DataFrame, tokenizer, train: bool, batch_size: int = 32) -> DataLoader:
    ds = EnsembleMultiSpectrumDataset(
        df, tokenizer,
        max_spectra_per_mol=CFG.MAX_SPECTRA_TRAIN if train else CFG.MAX_SPECTRA_INFER,
        is_train=train,
    )
    workers = CFG.NUM_WORKERS if train else 0
    return DataLoader(ds, batch_size=batch_size, shuffle=train, drop_last=train,
                      num_workers=workers, persistent_workers=workers > 0)


@lru_cache(maxsize=500_000)
def cached_ecfp4_fg71(smiles: str):
    return smiles_to_ecfp4_and_mass2smiles_71d(smiles)


def rank_ensemble_record(rec: Dict, reranker: "CASMI26EnsembleReranker", top_n_sub: int = 25) -> List[str]:
    """Blended score -> mass-consistent candidates first -> strict InChIKey14 dedup -> generated structure last."""
    blended = reranker.score_candidates_for_molecule(
        s_glmr_gen=rec["s_glmr_gen"], s_glmr_pre=rec["s_glmr_pre"], s_siamese_ecfp4=rec["s_siamese_ecfp4"],
        s_fg_l1_dist=rec["s_fg_l1_dist"], gen_smiles_valid=rec["gen_valid"],
    )
    ranked_order = np.lexsort((-blended, ~np.asarray(rec["in_window"], dtype=bool)))
    seen_ik14, top25_smiles = set(), []
    for r_i in ranked_order:
        c_smi, ik14 = rec["cand_canon_smis"][r_i], rec["cand_ik14s"][r_i]
        if c_smi is not None and ik14 is not None and ik14 not in seen_ik14:
            seen_ik14.add(ik14)
            top25_smiles.append(c_smi)
        if len(top25_smiles) >= top_n_sub:
            break
    # Append generated de novo structure if valid, distinct, and slots remain
    if rec["gen_valid"] and rec["gen_ik14"] not in seen_ik14 and len(top25_smiles) < top_n_sub:
        top25_smiles.append(rec["gen_smiles"])
    return top25_smiles[:top_n_sub]


@torch.no_grad()
def run_blended_casmi26_inference(
    glmr_model: nn.Module,
    siamese_model: SiameseECFP4Retriever,
    fg_head: Mass2SMILESFunctionalGroupHead,
    reranker: CASMI26EnsembleReranker,
    test_loader,
    tokenizer,
    candidate_smiles: List[str],
    candidate_masses: np.ndarray,
    candidate_glmr_embs: torch.Tensor,
    canonicalize_fn,
    retrieve_topk_fn,
    top_k: int = W.int("nb3.retrieval.top_k"),  # Expand pre-retrieval pool to K=60 before ensemble re-ranking
    top_n_sub: int = 25,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    collect_records: Optional[List[Dict]] = None,
) -> List[Dict[str, str]]:
    """
    Runs Stage 1 Pre-Retrieval (GLMR + Siamese ECFP4 union), Stage 2 GLMR Cross-Fusion
    Beam Search, Mass2SMILES 71-D Functional Group scoring, and 4-way Z-score/RRF blending.
    Per-candidate stream scores are appended to `collect_records` for out-of-fold weight fitting.
    """
    glmr_model.eval().to(device)
    siamese_model.eval().to(device)
    fg_head.eval().to(device)
    cos_mix = W("nb3.siamese.cos_mix")

    submission_rows = []

    for batch in test_loader:
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device)

        # 1. GLMR Stage 1 Multi-Spectrum Encoding & Top-K Pre-Retrieval
        h_s_mol, e_s_mol, _ = glmr_model.encode_molecule_spectra(batch)
        h_m_k, topk_indices_batch = retrieve_topk_fn(
            e_s_mol,
            batch["precursor_mz"],
            batch["primary_adduct"],
            candidate_masses,
            candidate_glmr_embs,
            top_k=top_k,
        )

        # 2. GLMR Stage 2 Cross-Fusion & Beam Search Generation
        h_ca = glmr_model.cross_fusion(h_s_mol, h_m_k)
        gen_token_seqs = glmr_model.mol_decoder.beam_search_decode(
            h_ca, beam_size=CFG.BEAM_SIZE, max_len=W.int("nb1.decode.max_len")
        )

        # 3. Mass2SMILES 71-D Functional Group + Heteroatom Prediction from H^s_mol
        fg_71_pred, _ = fg_head(h_s_mol)  # (B, 71)

        # 4. Siamese 2000-Bin Spectrum Encoding & Continuous ECFP4 Prediction
        z_s_batch, fp_logits_batch = siamese_model.encode_spectrum(batch["binned_2000"])
        pred_ecfp4_probs = torch.sigmoid(fp_logits_batch)  # (B, 2048)

        for b, mol_id in enumerate(batch["molecule_id"]):
            cand_indices = topk_indices_batch[b]
            cand_glmr_embs = h_m_k[b]  # (K, 256)
            neutral = neutral_mass_from_precursor(
                float(batch["precursor_mz"][b]), batch["primary_adduct"][b], batch["ionization_mode"][b]
            )

            # Stream A: GLMR Stage 1 Spectral-Molecular Cosine Similarity
            s_glmr_pre = torch.matmul(cand_glmr_embs, e_s_mol[b]).cpu().numpy()

            # Stream B: GLMR Stage 2 Unimodal Generative Cosine Similarity
            raw_gen_smi = tokenizer.decode(gen_token_seqs[b])
            canon_gen_smi, gen_ik14 = canonicalize_fn(raw_gen_smi)
            gen_valid = canon_gen_smi is not None

            if gen_valid:
                gen_tok = tokenizer.encode(canon_gen_smi, add_cls=True).unsqueeze(0).to(device)
                _, e_m_plus = glmr_model.mol_encoder(gen_tok)
                s_glmr_gen = torch.matmul(cand_glmr_embs, e_m_plus.squeeze(0)).cpu().numpy()
            else:
                s_glmr_gen = np.zeros_like(s_glmr_pre)

            # Extract ECFP4 & 71-D FG vectors for the K=60 pre-retrieved candidates
            cand_ecfp4_list, cand_fg71_list, cand_canon_smis, cand_ik14s = [], [], [], []
            for c_idx in cand_indices:
                smi = candidate_smiles[c_idx]
                c_smi, ik14 = canonicalize_fn(smi)
                ecfp4_vec, fg71_vec = cached_ecfp4_fg71(c_smi or smi)
                if ecfp4_vec is None:
                    ecfp4_vec = np.zeros(2048, dtype=np.float32)
                    fg71_vec = np.zeros(71, dtype=np.float32)
                cand_ecfp4_list.append(ecfp4_vec)
                cand_fg71_list.append(fg71_vec)
                cand_canon_smis.append(c_smi)
                cand_ik14s.append(ik14)

            cand_ecfp4_t = torch.tensor(np.stack(cand_ecfp4_list), dtype=torch.float32, device=device)
            cand_fg71_t = torch.tensor(np.stack(cand_fg71_list), dtype=torch.float32, device=device)

            # Stream C: 2000-Bin Siamese Cosine + Differentiable ECFP4 Tanimoto Similarity
            z_f_cands = siamese_model.encode_fingerprint(cand_ecfp4_t)  # (K, 256)
            siamese_cos = torch.matmul(z_f_cands, z_s_batch[b])          # (K,)
            ecfp4_tani = SiameseContrastiveTanimotoLoss.continuous_tanimoto_similarity(
                pred_ecfp4_probs[b].unsqueeze(0).expand_as(cand_ecfp4_t),
                cand_ecfp4_t,
            )
            s_siamese_ecfp4 = (cos_mix * siamese_cos + (1.0 - cos_mix) * ecfp4_tani).cpu().numpy()

            # Stream D: Mass2SMILES 71-D Functional Group + Heteroatom L1 Distance
            s_fg_l1_dist = torch.mean(
                torch.abs(cand_fg71_t - fg_71_pred[b].unsqueeze(0)), dim=-1
            ).cpu().numpy()

            rec = {
                "molecule_id": str(mol_id),
                "s_glmr_gen": s_glmr_gen, "s_glmr_pre": s_glmr_pre,
                "s_siamese_ecfp4": s_siamese_ecfp4, "s_fg_l1_dist": s_fg_l1_dist,
                "gen_valid": gen_valid, "gen_smiles": canon_gen_smi, "gen_ik14": gen_ik14,
                "cand_canon_smis": cand_canon_smis, "cand_ik14s": cand_ik14s,
                "in_window": in_mass_window(np.asarray(candidate_masses)[cand_indices], neutral),
            }
            if collect_records is not None:
                collect_records.append(rec)

            # Blend all 4 streams + Reciprocal Rank Fusion, strict InChIKey14 dedup up to 25 candidates
            top25_smiles = rank_ensemble_record(rec, reranker, top_n_sub)
            submission_rows.append(
                {
                    "molecule_id": str(mol_id),
                    "smiles": ";".join(top25_smiles[:top_n_sub]),
                }
            )

    return submission_rows

## Cell 6: Train the Heads, Fit the Blend Out-of-Fold, Infer the Test Set

The PDF provides the modules but no training or fitting run. This cell trains the Siamese ECFP4 learner and the Mass2SMILES head on notebook 1's training split (never its validation blocks), scores each stream and the blend on held-out timsTOF molecules, cross-fits `optimize_weights_oof` (fit on one half, score the other) so the reported gain is honest, and infers `test.parquet` with the blend `adopt` will activate: the fit if its cross-fitted MRR@25 beats the prior, otherwise the Evidently prior (`metrics.json` → `test_blend`). `fitted_weights.json` feeds `tools/evidently_weights.py adopt`, which applies the same rule. Runs on CPU.

In [ ]:
# [Colab Ensemble Cell 6] Train the Siamese ECFP4 and Mass2SMILES heads, fit blend weights out-of-fold, infer test
class EnsCFG:
    N_TRAIN_MOLECULES = int(os.environ.get("RL77_NB3_TRAIN_MOLECULES", 20000))  # subset of notebook 1's train split
    N_VAL_MOLECULES = int(os.environ.get("RL77_NB3_VAL_MOLECULES", 300))       # subset of notebook 1's val split
    SIAMESE_MINUTES = float(os.environ.get("RL77_SIAMESE_MINUTES", 20))
    FG_HEAD_MINUTES = float(os.environ.get("RL77_FG_HEAD_MINUTES", 30))
    BATCH_SIAMESE = 256
    BATCH_FG = 64
    LR = 1e-3


def molecule_features(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """Per inchikey14: composite 2000-bin spectrum, ECFP4 bits, 71-D Mass2SMILES target."""
    specs, fps, fgs, ids = [], [], [], []
    for ik14, grp in df.groupby("inchikey14", sort=False):
        records = grp.head(CFG.MAX_SPECTRA_TRAIN).to_dict("records")
        ecfp4, fg71 = cached_ecfp4_fg71(str(records[0]["normalized_smiles"]))
        if ecfp4 is None:
            continue
        specs.append(aggregate_molecule_2000_bins(records))
        fps.append(ecfp4)
        fgs.append(fg71)
        ids.append(ik14)
    return np.stack(specs).astype(np.float32), np.stack(fps).astype(np.float32), np.stack(fgs).astype(np.float32), ids


def train_siamese(spec: np.ndarray, fp: np.ndarray, deadline: "ENV.Deadline") -> Tuple[SiameseECFP4Retriever, List[float]]:
    model = SiameseECFP4Retriever().to(CFG.DEVICE)
    loss_fn = SiameseContrastiveTanimotoLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=EnsCFG.LR, weight_decay=1e-4)
    spec_t, fp_t = torch.from_numpy(spec), torch.from_numpy(fp)
    history, epoch = [], 0
    while not deadline.expired():
        model.train()
        perm = torch.randperm(len(spec_t))
        total, n = 0.0, 0
        for start in range(0, len(perm) - EnsCFG.BATCH_SIAMESE + 1, EnsCFG.BATCH_SIAMESE):  # drop_last for BatchNorm
            idx = perm[start:start + EnsCFG.BATCH_SIAMESE]
            xs, xf = spec_t[idx].to(CFG.DEVICE), fp_t[idx].to(CFG.DEVICE)
            z_s, z_f, fp_logits = model(xs, xf)
            loss = loss_fn(z_s, z_f, fp_logits, xf)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total, n = total + loss.item(), n + 1
            if deadline.expired():
                break
        history.append(total / max(n, 1))
        ENV.log(f"Siamese epoch {epoch}: loss {history[-1]:.4f} ({deadline.remaining_min():.1f} min left)")
        epoch += 1
    return model.eval(), history


def train_fg_head(glmr_model: GLMRPipeline, loader: DataLoader, deadline: "ENV.Deadline") -> Tuple[Mass2SMILESFunctionalGroupHead, List[float]]:
    """Mass2SMILES head on frozen GLMR multi-spectrum states H^s_mol (targets: log1p 71-D counts from SMILES)."""
    fg_head = Mass2SMILESFunctionalGroupHead().to(CFG.DEVICE)
    opt = torch.optim.AdamW(fg_head.parameters(), lr=EnsCFG.LR, weight_decay=1e-4)
    glmr_model.eval()
    history, epoch = [], 0
    while not deadline.expired():
        fg_head.train()
        total, n = 0.0, 0
        for batch in loader:
            if deadline.expired():
                break
            batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            targets = np.stack([cached_ecfp4_fg71(s)[1] if cached_ecfp4_fg71(s)[1] is not None else np.zeros(71, np.float32)
                                for s in batch["smiles"]])
            with torch.no_grad():
                h_s_mol, _, _ = glmr_model.encode_molecule_spectra(batch)
            fg_pred, _ = fg_head(h_s_mol)
            loss = mass2smiles_fg_loss(fg_pred, torch.from_numpy(targets).to(CFG.DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            total, n = total + loss.item(), n + 1
        history.append(total / max(n, 1))
        ENV.log(f"FG head epoch {epoch}: MAE+MSE {history[-1]:.4f} ({deadline.remaining_min():.1f} min left)")
        epoch += 1
    return fg_head.eval(), history


def ensemble_mrr(records: List[Dict], weights, gt: Dict[str, str]) -> float:
    reranker = CASMI26EnsembleReranker(weights=tuple(weights))
    preds = {r["molecule_id"]: rank_ensemble_record(r, reranker) for r in records}
    return compute_casmi_mrr25(preds, {r["molecule_id"]: gt[r["molecule_id"]] for r in records})


def crossfit_blend_weights(records: List[Dict], gt: Dict[str, str], prior: np.ndarray) -> Dict:
    """2-fold cross-fitting of optimize_weights_oof, then a final fit on all validation records."""
    rng = np.random.default_rng(CFG.SEED)
    order = rng.permutation(len(records))
    folds = [[records[i] for i in order[0::2]], [records[i] for i in order[1::2]]]
    held_total = 0.0
    for fit, held in ((folds[0], folds[1]), (folds[1], folds[0])):
        rr = CASMI26EnsembleReranker(weights=tuple(prior))
        w_fit = rr.optimize_weights_oof(fit, gt)
        held_total += ensemble_mrr(held, w_fit, gt) * len(held)
    final = CASMI26EnsembleReranker(weights=tuple(prior)).optimize_weights_oof(records, gt)
    return {
        "weights": [float(x) for x in final],
        "val_mrr25_prior": ensemble_mrr(records, prior, gt),
        "val_mrr25_crossfit": held_total / len(records),
        "val_mrr25_in_sample": ensemble_mrr(records, final, gt),
    }


def run_ensemble_pipeline() -> Dict:
    out_dir = ENV.work_dir()
    metrics: Dict = {"weights_version": W.version}
    nb1_dir = ENV.upstream_dir("nb1", "glmr_model.pt")
    split_dir = ENV.upstream_dir("nb1", "val_inchikey14.txt")
    glmr_model, tokenizer = load_glmr_checkpoint(nb1_dir / "glmr_model.pt", device=CFG.DEVICE)
    retriever = OfflineMassAndFormulaRetriever()

    def store_retrieve_fn(e_s_mol, precursor_mzs, adducts, candidate_masses, candidate_embs, top_k):
        return retriever.retrieve_top_k_batch(e_s_mol, precursor_mzs, adducts, top_k=top_k)

    # 1. Training molecules: a subset of notebook 1's training split (never the validation blocks)
    rng = np.random.default_rng(CFG.SEED)
    train_ids = read_split(split_dir, "train")
    train_ids = rng.choice(train_ids, size=min(EnsCFG.N_TRAIN_MOLECULES, len(train_ids)), replace=False).tolist()
    train_df = load_spectra_for_inchikeys(CFG.DATA_DIR, train_ids)
    spec, fp, _, ids = molecule_features(train_df)
    ENV.log(f"Ensemble training features: {len(ids):,} molecules")

    # 2. Siamese ECFP4 metric learner and Mass2SMILES functional-group head
    siamese, siamese_hist = train_siamese(spec, fp, ENV.Deadline(EnsCFG.SIAMESE_MINUTES))
    del spec, fp
    fg_head, fg_hist = train_fg_head(
        glmr_model, make_loader(train_df, tokenizer, train=True, batch_size=EnsCFG.BATCH_FG), ENV.Deadline(EnsCFG.FG_HEAD_MINUTES)
    )
    torch.save({"state_dict": siamese.state_dict(), "weights_version": W.version}, out_dir / "siamese_ecfp4.pt")
    torch.save({"state_dict": fg_head.state_dict(), "weights_version": W.version}, out_dir / "mass2smiles_fg_head.pt")
    metrics["train"] = {"siamese_last_loss": siamese_hist[-1] if siamese_hist else None,
                        "fg_head_last_loss": fg_hist[-1] if fg_hist else None,
                        "siamese_epochs": len(siamese_hist), "fg_head_epochs": len(fg_hist)}

    # 3. Validation records on held-out timsTOF molecules -> per-stream and blended MRR@25, cross-fitted weights
    val_ids = read_split(split_dir, "val")
    val_ids = rng.choice(val_ids, size=min(EnsCFG.N_VAL_MOLECULES, len(val_ids)), replace=False).tolist()
    val_df = load_spectra_for_inchikeys(CFG.DATA_DIR, val_ids, timstof_window_only=True)
    records: List[Dict] = []
    run_blended_casmi26_inference(
        glmr_model, siamese, fg_head, CASMI26EnsembleReranker(), make_ensemble_loader(val_df, tokenizer, train=False, batch_size=16),
        tokenizer, retriever.smiles_arr, retriever.masses_arr, None, canonicalize_and_inchikey14, store_retrieve_fn,
        device=CFG.DEVICE, collect_records=records,
    )
    gt = ground_truth_ik14(val_df)
    prior = np.asarray(W.vec("nb3.reranker", ["w_gen", "w_pre", "w_siamese", "w_fg", "w_rrf"]))
    streams = {"gen": 0, "pre": 1, "siamese": 2, "fg": 3, "rrf": 4}
    metrics["val_stream_mrr25"] = {name: ensemble_mrr(records, np.eye(5)[i], gt) for name, i in streams.items()}
    fit = crossfit_blend_weights(records, gt, prior)
    metrics["val_blend"] = {k: v for k, v in fit.items() if k != "weights"}
    metrics["val_blend"]["n"] = len(records)
    ENV.log(f"Validation per-stream MRR@25 {metrics['val_stream_mrr25']}; blend {metrics['val_blend']}")
    ENV.write_json(out_dir / "fitted_weights.json", {
        "notebook": "nb3",
        "weights": {f"nb3.reranker.{n}": w for n, w in zip(["w_gen", "w_pre", "w_siamese", "w_fg", "w_rrf"], fit["weights"])},
        "evidence": {"metric": "MRR@25", "split": f"{len(records)} held-out timsTOF molecules, 2-fold cross-fit",
                     "prior": fit["val_mrr25_prior"], "fitted_crossfit": fit["val_mrr25_crossfit"]},
    })

    # 4. Test inference with the blend `adopt` will activate: the fit only if its cross-fitted MRR@25 beats the prior
    blend, used_fit = ENV.select_validated(prior, np.asarray(fit["weights"]), fit["val_mrr25_prior"], fit["val_mrr25_crossfit"])
    metrics["test_blend"] = {"weights": [float(x) for x in blend], "source": "fitted" if used_fit else W.version}
    ENV.log(f"Test blend ({metrics['test_blend']['source']}): {metrics['test_blend']['weights']}")
    test_df = pd.read_parquet(CFG.DATA_DIR / "test.parquet")
    rows = run_blended_casmi26_inference(
        glmr_model, siamese, fg_head, CASMI26EnsembleReranker(weights=tuple(blend)),
        make_ensemble_loader(test_df, tokenizer, train=False, batch_size=16),
        tokenizer, retriever.smiles_arr, retriever.masses_arr, None, canonicalize_and_inchikey14, store_retrieve_fn,
        device=CFG.DEVICE,
    )
    sub_df = pd.DataFrame(rows, columns=["molecule_id", "smiles"])
    validate_submission_file(sub_df, test_df["molecule_id"].unique().tolist())
    sub_df.to_csv(out_dir / "submission.csv", index=False)

    ENV.write_json(out_dir / "metrics.json", metrics)
    ENV.write_run_manifest("nb3", out_dir, W.version, {"metrics": metrics})
    ENV.log(f"Notebook 3 outputs written to {out_dir}")
    return metrics


if __name__ == "__main__":
    NB3_METRICS = run_ensemble_pipeline()

## Why This Four-Way Ensemble Lifts MRR@25 Across All Three Novelty Tiers

| Candidate Scenario in `test.parquet` | Primary Scoring Stream Driving Top-1 Rank | Why Single-Model Pipelines Fail |
| --- | --- | --- |
| **Tier 1: In-Library Match** | `s_siamese_ecfp4` + `s_glmr_pre` | Dense 2000-bin vectors capture full fragment ratios without peak-count truncation loss. |
| **Tier 2: Known DB Isomers (PubChem/COCONUT)** | `s_glmr_gen` + `s_fg_match` (Mass2SMILES) | Positional isomers share almost identical ECFP4 bits; 60 functional group & sugar counts break ties. |
| **Tier 3: Novel Scaffolds / Low-Signal Spectra** | `z_rrf` + `s_siamese_ecfp4` fallback | When autoregressive beam search produces an invalid SMILES, RRF seamlessly shifts weight to Siamese + FG streams. |

The table above illustrates how each branch compensates for the others' failure modes while computing re-ranking features only on the $K = 60$ pre-retrieved candidates per molecule, adding less than 15 seconds of total wall-clock overhead across all 400 test molecules. *(PDF claims; the measured per-stream MRR@25 is in `metrics.json`.)*